In [ ]:
import os
# Set WANDB_API_KEY as an environment variable (never commit an API key).

# Ορισμός TensorBoard Log Directory μέσω Environment Variable
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"

In [ ]:
!pip install wandb -qU  # Εγκατάσταση της πιο πρόσφατης έκδοσης της βιβλιοθήκης WandB.
import wandb            # Εισαγωγή της WandB.

# Αυτή η εντολή ενεργοποιεί τον χρήστη να συνδεθεί στο λογαριασμό του WandB.
# Θα χρειαστεί ένα API Key.
wandb.login()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from __future__ import annotations

import json
import re
import time
import unicodedata
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from difflib import SequenceMatcher
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import pandas as pd
import requests

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

BASE_DIR = Path('/content/drive/MyDrive/ΔΙΔΑΚΤΟΡΙΚΟ PhD/4o PAPER/athar_acl_ocl_outputs')

FINAL_INPUT_CSV = BASE_DIR / 'final_citation_non_citation_dataset.csv'
ATHAR_INPUT_CSV = BASE_DIR / 'athar_citation_sentences.csv'

# Requested main output name.
OUTPUT_CSV = BASE_DIR / '_final_citation_non_citation_dataset.csv'

# Audit/support files. These do not change the main schema.
AUDIT_CSV = BASE_DIR / '_final_citation_non_citation_dataset_target_audit.csv'
UNRESOLVED_CSV = BASE_DIR / '_final_citation_non_citation_dataset_unresolved.csv'
METADATA_CACHE_JSON = BASE_DIR / 'acl_target_metadata_cache.json'

TARGET_TOKEN = '[TARGET_CITATION]'
OTHER_TOKEN = '[OTHER_CITATION]'

In [ ]:
# Strict mode means that an Athar citation row is written to the main output
# only when the target marker is uniquely resolved and the citation markers
# can be transformed safely. Unresolved Athar rows are written separately.
STRICT_MODE = True

REQUEST_TIMEOUT = 30
REQUEST_RETRIES = 4
REQUEST_SLEEP_SECONDS = 0.08

REQUIRED_FINAL_COLUMNS = [
    'Source_Paper_ID',
    'sentence',
    'is_citation',
    'polarity',
    'source',
    'sampling_status',
]

REQUIRED_ATHAR_COLUMNS = [
    'Source_Paper_ID',
    'Target_Paper_ID',
    'sentence',
    'is_citation',
    'polarity',
    'source',
    'sampling_status',
]

JOIN_COLUMNS = [
    'Source_Paper_ID',
    'sentence',
    'is_citation',
    'polarity',
    'source',
    'sampling_status',
]

In [ ]:
# ============================================================
# TEXT NORMALIZATION
# ============================================================

def normalize_for_matching(text: str) -> str:
    text = '' if text is None else str(text)
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    text = text.replace('’', "'").replace('`', "'")
    text = re.sub(r'[^A-Za-z0-9]+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip().casefold()


def surname_aliases(family_name: str) -> List[str]:
    norm = normalize_for_matching(family_name)
    if not norm:
        return []
    parts = norm.split()
    aliases = {norm, parts[-1]}
    if len(parts) >= 2:
        aliases.add(' '.join(parts[-2:]))
    return sorted(a for a in aliases if a)

In [ ]:
# ============================================================
# ACL ANTHOLOGY METADATA
# Official canonical IDs support per-paper .xml files.
# ============================================================

def load_metadata_cache(path: Path) -> Dict[str, dict]:
    if path.exists():
        with path.open('r', encoding='utf-8') as f:
            return json.load(f)
    return {}


def save_metadata_cache(cache: Dict[str, dict], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)


def parse_mods_xml(xml_text: str, acl_id: str) -> dict:
    ns = {'mods': 'http://www.loc.gov/mods/v3'}
    root = ET.fromstring(xml_text)

    mods = root if root.tag.endswith('mods') else root.find('mods:mods', ns)
    if mods is None:
        raise ValueError(f'No MODS record found for {acl_id}')

    title_node = mods.find('mods:titleInfo/mods:title', ns)
    title = title_node.text.strip() if title_node is not None and title_node.text else ''

    date_node = mods.find('mods:originInfo/mods:dateIssued', ns)
    date_issued = date_node.text.strip() if date_node is not None and date_node.text else ''
    year_match = re.search(r'(?:19|20)\d{2}', date_issued)
    year = year_match.group(0) if year_match else ''

    authors = []
    for name in mods.findall('mods:name[@type="personal"]', ns):
        role_terms = [
            (n.text or '').strip().casefold()
            for n in name.findall('mods:role/mods:roleTerm', ns)
        ]
        if role_terms and 'author' not in role_terms:
            continue

        family_node = name.find('mods:namePart[@type="family"]', ns)
        given_nodes = name.findall('mods:namePart[@type="given"]', ns)
        family = family_node.text.strip() if family_node is not None and family_node.text else ''
        given = ' '.join((n.text or '').strip() for n in given_nodes if (n.text or '').strip())
        if family:
            authors.append({'family': family, 'given': given})

    if not authors or not year:
        raise ValueError(f'Incomplete ACL metadata for {acl_id}: authors={authors}, year={year}')

    return {
        'acl_id': acl_id,
        'title': title,
        'year': year,
        'authors': authors,
        'url': f'https://aclanthology.org/{acl_id}/',
    }


def fetch_acl_metadata(acl_id: str, session: requests.Session) -> dict:
    url = f'https://aclanthology.org/{acl_id}.xml'
    last_error = None
    for attempt in range(1, REQUEST_RETRIES + 1):
        try:
            response = session.get(url, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            return parse_mods_xml(response.text, acl_id)
        except Exception as exc:
            last_error = exc
            if attempt < REQUEST_RETRIES:
                time.sleep(min(2 ** (attempt - 1), 8))
    raise RuntimeError(f'Could not fetch metadata for {acl_id}: {last_error}')


def get_all_acl_metadata(target_ids: List[str], cache_path: Path) -> Dict[str, dict]:
    cache = load_metadata_cache(cache_path)
    session = requests.Session()
    session.headers.update({'User-Agent': 'target-aware-citation-preprocessing/1.0 (research use)'})

    missing = [tid for tid in target_ids if tid not in cache]
    print(f'ACL metadata: {len(target_ids)} unique targets; {len(missing)} need download.')

    failures = []
    for i, acl_id in enumerate(missing, start=1):
        try:
            cache[acl_id] = fetch_acl_metadata(acl_id, session)
        except Exception as exc:
            failures.append((acl_id, str(exc)))
            cache[acl_id] = {'acl_id': acl_id, 'error': str(exc)}

        if i % 20 == 0 or i == len(missing):
            save_metadata_cache(cache, cache_path)
            print(f'  metadata progress: {i}/{len(missing)}')
        time.sleep(REQUEST_SLEEP_SECONDS)

    save_metadata_cache(cache, cache_path)

    if failures:
        print(f'WARNING: metadata failed for {len(failures)} target IDs.')
        for acl_id, error in failures[:10]:
            print(f'  {acl_id}: {error}')

    return cache

In [ ]:
# ============================================================
# CITATION MARKER PARSING
# The unit is an atomic citation occurrence.
# ============================================================

# Broad author-year grammar designed for noisy Athar citation contexts.
PARTICLE_SURNAME = r'(?:van|von|de|del|der|den|da|di|la|le)\s+[A-ZÀ-ÖØ-Þ][A-Za-zÀ-ÖØ-öø-ÿ\'’\-]+'
PLAIN_SURNAME = r'[A-ZÀ-ÖØ-Þ][A-Za-zÀ-ÖØ-öø-ÿ\'’\-]+'
SURNAME = rf'(?:{PARTICLE_SURNAME}|{PLAIN_SURNAME})'

AUTHOR_LIST = rf'''
(?:
    {SURNAME}
    (?:
        \s+(?:et\s+al\.?)
      | (?:\s*,\s*{SURNAME}){{0,5}}\s*,?\s*(?:and|&)\s*{SURNAME}
      | (?:\s*,\s*{SURNAME}){{1,6}}
    )?
)
'''

YEAR_PATTERN = r'(?:19|20)\d{2}[a-z]?'

AUTHOR_YEAR_RE = re.compile(
    rf'''(?P<authors>{AUTHOR_LIST})
        \s*,?\s*
        (?P<open>[\(\[])?\s*
        (?P<year>{YEAR_PATTERN})
        \s*(?P<close>[\)\]])?''',
    re.X,
)

CONTINUATION_YEAR_RE = re.compile(rf'\s*,\s*(?P<year>{YEAR_PATTERN})')

# Numeric citation markers, deliberately limited to <=3 digits so [1993]
# is treated as an author-year year rather than reference number 1993.
NUMERIC_CLUSTER_RE = re.compile(
    r'\[(?P<body>\s*\d{1,3}[a-z]?(?:\s*(?:,|;|–|-|—)\s*\d{1,3}[a-z]?)*\s*)\]'
)

In [ ]:
@dataclass
class CitationAtom:
    start: int
    end: int
    surface: str
    style: str
    year: str = ''
    authors_text: str = ''
    inherited_authors_text: str = ''

    @property
    def match_authors_text(self) -> str:
        return self.authors_text or self.inherited_authors_text


def extract_author_year_atoms(sentence: str) -> List[CitationAtom]:
    atoms: List[CitationAtom] = []

    for m in AUTHOR_YEAR_RE.finditer(sentence):
        start, end = m.span()
        open_char = m.group('open') or ''
        close_char = m.group('close') or ''

        # If a closing parenthesis/bracket belongs to the outer citation cluster
        # (e.g. "... Cutting et al., 1992)"), preserve it. If this match itself
        # opened the year delimiter (e.g. "Jing (2000)"), keep the pair.
        if close_char and not open_char:
            end -= 1

        if end <= start:
            continue

        atom = CitationAtom(
            start=start,
            end=end,
            surface=sentence[start:end],
            style='author_year',
            year=m.group('year'),
            authors_text=m.group('authors'),
        )
        atoms.append(atom)

        # Handle compressed repeated-year citations such as:
        # "Yarowsky 1994, 1995" or "Zhang et al., 2007, 2008a, 2008b".
        pos = m.end()
        while True:
            cm = CONTINUATION_YEAR_RE.match(sentence, pos)
            if cm is None:
                break
            ys, ye = cm.span('year')
            atoms.append(
                CitationAtom(
                    start=ys,
                    end=ye,
                    surface=sentence[ys:ye],
                    style='author_year_continuation',
                    year=cm.group('year'),
                    inherited_authors_text=m.group('authors'),
                )
            )
            pos = cm.end()

    # Remove duplicate/overlapping atoms, preferring the longer first-author match.
    atoms.sort(key=lambda a: (a.start, -(a.end - a.start)))
    kept: List[CitationAtom] = []
    for atom in atoms:
        if any(not (atom.end <= k.start or atom.start >= k.end) for k in kept):
            continue
        kept.append(atom)
    return sorted(kept, key=lambda a: a.start)


def extract_numeric_atoms(sentence: str, occupied: List[Tuple[int, int]]) -> List[CitationAtom]:
    atoms: List[CitationAtom] = []
    for m in NUMERIC_CLUSTER_RE.finditer(sentence):
        if any(not (m.end() <= s or m.start() >= e) for s, e in occupied):
            continue

        body = m.group('body')
        body_start = m.start('body')
        # Split comma/semicolon lists into atomic numbers. Ranges are retained as
        # one unresolved cluster because mapping individual references in a range
        # requires bibliography information.
        if re.search(r'[–—-]', body):
            atoms.append(
                CitationAtom(
                    start=m.start(),
                    end=m.end(),
                    surface=m.group(0),
                    style='numeric_range_cluster',
                )
            )
            continue

        for num in re.finditer(r'\d{1,3}[a-z]?', body):
            s = body_start + num.start()
            e = body_start + num.end()
            atoms.append(
                CitationAtom(
                    start=s,
                    end=e,
                    surface=sentence[s:e],
                    style='numeric',
                )
            )
    return atoms


def extract_all_atoms(sentence: str) -> List[CitationAtom]:
    author_year = extract_author_year_atoms(sentence)
    occupied = [(a.start, a.end) for a in author_year]
    numeric = extract_numeric_atoms(sentence, occupied)
    atoms = author_year + numeric
    atoms.sort(key=lambda a: a.start)
    return atoms


def fuzzy_ratio(a: str, b: str) -> float:
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()


def author_match_score(candidate_authors: str, metadata_authors: List[dict]) -> float:
    cand = normalize_for_matching(candidate_authors)
    if not cand or not metadata_authors:
        return 0.0

    first_family = metadata_authors[0].get('family', '')
    aliases = surname_aliases(first_family)
    score = 0.0

    # First author is the primary signal.
    if any(alias and re.search(rf'\b{re.escape(alias)}\b', cand) for alias in aliases):
        score += 70.0
    else:
        cand_tokens = cand.split()
        best = 0.0
        for alias in aliases:
            for token in cand_tokens:
                best = max(best, fuzzy_ratio(alias, token))
        if best >= 0.84:
            score += 55.0
        elif best >= 0.76:
            score += 40.0

    # Additional family names improve confidence but are not mandatory because
    # citations often use "et al.".
    for author in metadata_authors[1:4]:
        fam_aliases = surname_aliases(author.get('family', ''))
        if any(alias and re.search(rf'\b{re.escape(alias)}\b', cand) for alias in fam_aliases):
            score += 8.0

    if 'et al' in cand and len(metadata_authors) >= 3:
        score += 5.0

    return min(score, 100.0)


def find_target_atom(
    atoms: List[CitationAtom], metadata: dict
) -> Tuple[Optional[int], str, float]:
    if not metadata or metadata.get('error'):
        return None, 'metadata_unavailable', 0.0

    target_year = str(metadata.get('year', ''))
    target_authors = metadata.get('authors', [])
    if not target_year or not target_authors:
        return None, 'metadata_incomplete', 0.0

    scored = []
    for idx, atom in enumerate(atoms):
        if not atom.year:
            continue
        # Match publication year, ignoring suffix a/b/c for the metadata year.
        if atom.year[:4] != target_year[:4]:
            continue
        a_score = author_match_score(atom.match_authors_text, target_authors)
        if a_score > 0:
            scored.append((idx, a_score))

    if not scored:
        return None, 'target_not_found', 0.0

    scored.sort(key=lambda x: x[1], reverse=True)
    best_idx, best_score = scored[0]

    # Minimum conservative confidence for first-author/year resolution.
    if best_score < 55.0:
        return None, 'target_low_confidence', best_score

    # If more than one occurrence has the same strong score, the raw sentence
    # alone does not tell us which occurrence the Athar row intended.
    strong_near_best = [x for x in scored if x[1] >= max(55.0, best_score - 3.0)]
    if len(strong_near_best) > 1:
        return None, 'multiple_target_occurrences', best_score

    return best_idx, 'resolved', best_score


def uncovered_years(sentence: str, atoms: List[CitationAtom]) -> List[str]:
    covered = [(a.start, a.end) for a in atoms if a.year]
    residual = []
    for m in re.finditer(YEAR_PATTERN, sentence):
        if not any(m.start() >= s and m.end() <= e for s, e in covered):
            residual.append(m.group(0))
    return residual


def remove_redundant_single_citation_wrappers(text: str) -> str:
    """
    Remove parentheses/brackets only when they contain exactly one
    TARGET/OTHER citation token.

    Examples:
        ([TARGET_CITATION]) -> [TARGET_CITATION]
        ([OTHER_CITATION])  -> [OTHER_CITATION]
        [[TARGET_CITATION]] -> [TARGET_CITATION]

    Citation clusters are preserved:
        ([TARGET_CITATION]; [OTHER_CITATION])
        [[TARGET_CITATION], [OTHER_CITATION]]
    """

    tokens = [TARGET_TOKEN, OTHER_TOKEN]

    for token in tokens:
        escaped = re.escape(token)

        # Standalone author-year citation:
        # ([TARGET_CITATION]) -> [TARGET_CITATION]
        text = re.sub(
            rf'\(\s*{escaped}\s*\)',
            token,
            text
        )

        # Standalone numeric citation:
        # [[TARGET_CITATION]] -> [TARGET_CITATION]
        text = re.sub(
            rf'\[\s*{escaped}\s*\]',
            token,
            text
        )

    return text


def transform_sentence(
    sentence: str,
    metadata: dict,
) -> Tuple[str, dict]:
    atoms = extract_all_atoms(sentence)
    target_idx, target_status, target_score = find_target_atom(atoms, metadata)
    residual_years = uncovered_years(sentence, atoms)

    audit = {
        'status': target_status,
        'target_score': target_score,
        'n_detected_atoms': len(atoms),
        'n_author_year_atoms': sum(a.style.startswith('author_year') for a in atoms),
        'n_numeric_atoms': sum(a.style.startswith('numeric') for a in atoms),
        'residual_years': '|'.join(residual_years),
        'target_surface': '',
        'target_start': '',
        'target_end': '',
    }

    if target_idx is None:
        return sentence, audit

    if residual_years:
        audit['status'] = 'partial_marker_parse'
        return sentence, audit

    # Numeric ranges cannot be split to atomic cited works without bibliography.
    if any(a.style == 'numeric_range_cluster' for a in atoms):
        audit['status'] = 'unresolved_numeric_range'
        return sentence, audit

    target_atom = atoms[target_idx]
    audit['target_surface'] = target_atom.surface
    audit['target_start'] = target_atom.start
    audit['target_end'] = target_atom.end

    # Replace from right to left so character offsets stay valid.
    """transformed = sentence
    for idx in range(len(atoms) - 1, -1, -1):
        atom = atoms[idx]
        token = TARGET_TOKEN if idx == target_idx else OTHER_TOKEN
        transformed = transformed[:atom.start] + token + transformed[atom.end:]
    """
    # Replace from right to left so character offsets stay valid.
    transformed = sentence
    for idx in range(len(atoms) - 1, -1, -1):
        atom = atoms[idx]
        token = TARGET_TOKEN if idx == target_idx else OTHER_TOKEN
        transformed = transformed[:atom.start] + token + transformed[atom.end:]

    # Remove redundant wrappers around standalone citation tokens.
    # Example:
    # ([TARGET_CITATION]) -> [TARGET_CITATION]
    # But clusters such as
    # ([TARGET_CITATION]; [OTHER_CITATION])
    # are preserved.
    transformed = remove_redundant_single_citation_wrappers(transformed)


    # Final sanity checks.
    if transformed.count(TARGET_TOKEN) != 1:
        audit['status'] = 'target_token_count_error'
        return sentence, audit

    if re.search(YEAR_PATTERN, transformed):
        # A leftover year after replacement usually means a citation marker was
        # not fully parsed. Be conservative rather than mixing raw and special tokens.
        audit['status'] = 'leftover_year_after_transform'
        return sentence, audit

    audit['status'] = 'resolved'
    return transformed, audit

In [ ]:
# ============================================================
# SAFE 1:1 TARGET_PAPER_ID ATTACHMENT
# ============================================================

def attach_target_ids(final_df: pd.DataFrame, athar_df: pd.DataFrame) -> pd.DataFrame:
    final_df = final_df.copy()
    final_df['_original_row_order'] = range(len(final_df))

    final_athar = final_df[final_df['source'].eq('Athar')].copy()
    aux = athar_df.copy()

    if len(final_athar) != len(aux):
        raise ValueError(
            f'Athar row-count mismatch: final={len(final_athar)}, auxiliary={len(aux)}'
        )

    # Verify that the two Athar tables contain exactly the same multiset of rows
    # when Target_Paper_ID is ignored.
    final_counts = final_athar.groupby(JOIN_COLUMNS, dropna=False).size().rename('n_final')
    aux_counts = aux.groupby(JOIN_COLUMNS, dropna=False).size().rename('n_aux')
    count_check = final_counts.to_frame().join(aux_counts, how='outer').fillna(0)
    if not (count_check['n_final'].eq(count_check['n_aux'])).all():
        bad = count_check[count_check['n_final'].ne(count_check['n_aux'])].head(10)
        raise ValueError(
            'The Athar rows in the two files are not the same multiset. '\
            f'Examples:\n{bad}'
        )

    # Some identical Athar rows correspond to different Target_Paper_ID values.
    # A simple merge would create a many-to-many expansion. Add an occurrence
    # index within each exact duplicate group to force a safe 1:1 pairing.
    final_athar['_dup_occurrence'] = final_athar.groupby(
        JOIN_COLUMNS, dropna=False, sort=False
    ).cumcount()
    aux['_dup_occurrence'] = aux.groupby(
        JOIN_COLUMNS, dropna=False, sort=False
    ).cumcount()

    lookup_cols = JOIN_COLUMNS + ['_dup_occurrence', 'Target_Paper_ID']
    merged = final_athar.merge(
        aux[lookup_cols],
        on=JOIN_COLUMNS + ['_dup_occurrence'],
        how='left',
        validate='one_to_one',
    )

    if merged['Target_Paper_ID'].isna().any():
        raise ValueError('Some Athar rows could not be assigned Target_Paper_ID.')

    target_map = merged.set_index('_original_row_order')['Target_Paper_ID']
    final_df['Target_Paper_ID'] = final_df['_original_row_order'].map(target_map)
    return final_df

In [ ]:
# ============================================================
# MAIN
# ============================================================

def main() -> None:
    print('Loading CSV files...')
    final_df = pd.read_csv(FINAL_INPUT_CSV)
    athar_df = pd.read_csv(ATHAR_INPUT_CSV)

    missing_final = [c for c in REQUIRED_FINAL_COLUMNS if c not in final_df.columns]
    missing_athar = [c for c in REQUIRED_ATHAR_COLUMNS if c not in athar_df.columns]
    if missing_final:
        raise ValueError(f'Missing columns in final input: {missing_final}')
    if missing_athar:
        raise ValueError(f'Missing columns in Athar input: {missing_athar}')

    print(f'final rows: {len(final_df):,}')
    print(f'Athar rows in final: {(final_df["source"] == "Athar").sum():,}')
    print(f'non-Athar rows: {(final_df["source"] != "Athar").sum():,}')

    print('\nAttaching Target_Paper_ID safely...')
    work_df = attach_target_ids(final_df, athar_df)

    target_ids = sorted(
        work_df.loc[work_df['source'].eq('Athar'), 'Target_Paper_ID']
        .dropna().astype(str).unique().tolist()
    )
    print(f'Unique Athar target papers: {len(target_ids):,}')

    print('\nFetching/caching ACL Anthology metadata...')
    metadata_cache = get_all_acl_metadata(target_ids, METADATA_CACHE_JSON)

    new_sentences = []
    audit_rows = []

    print('\nTransforming sentences...')
    for processed, (idx, row) in enumerate(work_df.iterrows(), start=1):
        sentence = '' if pd.isna(row['sentence']) else str(row['sentence'])

        if row['source'] != 'Athar' or int(row['is_citation']) != 1:
            new_sentences.append(sentence)
            audit_rows.append({
                'original_row_index': int(row['_original_row_order']),
                'Source_Paper_ID': row['Source_Paper_ID'],
                'Target_Paper_ID': '',
                'status': 'noncitation_unchanged',
                'target_score': '',
                'target_surface': '',
                'target_start': '',
                'target_end': '',
                'n_detected_atoms': 0,
                'n_author_year_atoms': 0,
                'n_numeric_atoms': 0,
                'residual_years': '',
                'original_sentence': sentence,
                'transformed_sentence': sentence,
            })
            continue

        target_id = str(row['Target_Paper_ID'])
        metadata = metadata_cache.get(target_id, {'error': 'not_in_cache'})
        transformed, audit = transform_sentence(sentence, metadata)
        new_sentences.append(transformed)

        audit_rows.append({
            'original_row_index': int(row['_original_row_order']),
            'Source_Paper_ID': row['Source_Paper_ID'],
            'Target_Paper_ID': target_id,
            **audit,
            'metadata_year': metadata.get('year', ''),
            'metadata_first_author': (
                metadata.get('authors', [{}])[0].get('family', '')
                if metadata.get('authors') else ''
            ),
            'original_sentence': sentence,
            'transformed_sentence': transformed,
        })

        if processed % 1000 == 0:
            print(f'  processed {processed:,}/{len(work_df):,}')

    work_df['sentence'] = new_sentences
    audit_df = pd.DataFrame(audit_rows)

    # Main output keeps the original six-column schema only.
    resolved_indices = set(
        audit_df.loc[
            audit_df['status'].isin(['resolved', 'noncitation_unchanged']),
            'original_row_index'
        ].astype(int).tolist()
    )

    if STRICT_MODE:
        output_df = work_df[work_df['_original_row_order'].isin(resolved_indices)].copy()
    else:
        output_df = work_df.copy()

    output_df = output_df.sort_values('_original_row_order')
    output_df = output_df[REQUIRED_FINAL_COLUMNS]

    unresolved_df = audit_df[
        ~audit_df['status'].isin(['resolved', 'noncitation_unchanged'])
    ].copy()

    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    output_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
    audit_df.to_csv(AUDIT_CSV, index=False, encoding='utf-8-sig')
    unresolved_df.to_csv(UNRESOLVED_CSV, index=False, encoding='utf-8-sig')

    print('\n================ FINAL REPORT ================')
    print(f'Input rows:                     {len(final_df):,}')
    print(f'Resolved Athar target rows:     {(audit_df["status"] == "resolved").sum():,}')
    print(f'Unresolved Athar rows:          {len(unresolved_df):,}')
    print(f'Non-citation rows unchanged:    {(audit_df["status"] == "noncitation_unchanged").sum():,}')
    print(f'Rows in main output:            {len(output_df):,}')
    print('\nStatus counts:')
    print(audit_df['status'].value_counts(dropna=False).to_string())

    # Strong validation of the main output.
    citation_out = output_df[output_df['is_citation'].eq(1)]
    bad_target_count = citation_out['sentence'].astype(str).map(
        lambda s: s.count(TARGET_TOKEN) != 1
    )
    if bad_target_count.any():
        raise AssertionError(
            f'{bad_target_count.sum()} citation rows in main output do not contain exactly one {TARGET_TOKEN}.'
        )

    noncit_out = output_df[output_df['is_citation'].eq(0)]
    if noncit_out['sentence'].astype(str).str.contains(
        re.escape(TARGET_TOKEN) + '|' + re.escape(OTHER_TOKEN), regex=True
    ).any():
        raise AssertionError('Special citation tokens unexpectedly found in non-citation rows.')

    print('\nSaved:')
    print('MAIN:      ', OUTPUT_CSV)
    print('AUDIT:     ', AUDIT_CSV)
    print('UNRESOLVED:', UNRESOLVED_CSV)
    print('CACHE:     ', METADATA_CACHE_JSON)


if __name__ == '__main__':
    main()